# Meeting 1: train

Trains `m1-real`, `m1-oversample`, `m1-augment` and `m1-augment-strong` with 5 seeds each on the GPU. After each run, `model.pt` files go to Google Drive and everything else is pushed to GitHub.

Afterwards, read the validation results and set `AUGMENT_RUN` in `m1_test.ipynb`.

In [1]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/1"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes


<module 'colab_tools' from '/content/dane_syntetyczne/colab/colab_tools.py'>

In [2]:
RUNS = {  # name: (method, environment, tags)
    "m1-real": ("real", {}, []),
    "m1-oversample": ("oversample", {}, []),
    "m1-augment": ("augment", {"SYNTHDATA_AUGMENT": "mild"}, ["augment=mild"]),
    "m1-augment-strong": ("augment", {"SYNTHDATA_AUGMENT": "strong"}, ["augment=strong"]),
}

ct.setup()
ct.mount_drive()
for name, (method, env, tags) in RUNS.items():
    ct.train_run(name, method, env=env, tags=tags)
    ct.check_devices([name])
    ct.save_models(name)  # model.pt -> Drive
    ct.save_to_github(f"Meeting 1: train {name} in Colab")  # everything else -> git

$ git pull --ff-only --quiet
$ /usr/bin/python3 -m pip install --quiet -e .
Data: 905 course photos in ./data
GPU: Tesla T4
GPU: Tesla T4

=== train m1-real ===
$ synth-train --name m1-real --method real --device cuda --workers 2
m1-real: real, efficientnet_b0, 5 seeds, cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth

  0%|          | 0.00/20.5M [00:00<?, ?B/s]
 56%|█████▌    | 11.5M/20.5M [00:00<00:00, 120MB/s]
100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]
  seed 0: Egyptian Mau recall 0.450, macro-F1 0.860 (31 s)
  seed 1: Egyptian Mau recall 0.500, macro-F1 0.869 (24 s)
  seed 2: Egyptian Mau recall 0.450, macro-F1 0.861 (21 s)
  seed 3: Egyptian Mau recall 0.450, macro-F1 0.855 (23 s)
  seed 4: Egyptian Mau recall 0.600, macro-F1 0.877 (23 s)
Results of every seed are in /content/dane_syntetyczne/runs/m1-real/results.json
All 5 seeds trained on cuda.
m1-